<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">构建推理模型（从零开始）</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 E：批处理与吞吐量导向的执行

本笔记本中正在使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- 在主要章节中，我们通常一次处理一个示例
- 这使得代码保持紧凑且更易于理解
- 但同时，代码运行成本已经很高，因此由于硬件和资源限制，添加批处理支持带来的收益有限
- 然而在某些场景下，能够以批处理模式运行代码仍然很有用
- 本附录解释了批处理执行的基本概念，并展示了如何使用补充材料中的代码为不同章节应用此功能

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 为什么批处理有帮助

- 存在两种不同的性能目标：
  - 延迟：针对单个提示词获取答案的速度
  - 吞吐量：在给定时间内能够处理的提示词数量
- 单样本生成通常最适用于最小化延迟和代码调试
- 批处理主要针对吞吐量优化
- 若需在MATH-500数据集上评估数百个问题、生成大量自一致性样本，或在众多监督样本上进行训练，批处理能在合适硬件上显著缩短总运行时间
  - 需注意的是，批处理并非在所有设备上都能保证提速
  - 在CPU或部分优化不足的GPU上运行小模型时，批处理可能无法带来收益；由于额外填充和批处理开销可能抵消并行化优势，甚至可能导致速度下降

&nbsp;
## E.2 运行批量生成

- 批处理的主要技术障碍在于提示词通常长度各异
- 例如，一道数学题可能被分词为40个token，而另一道可能需要120个token
- 由于PyTorch中的张量必须是矩形形状，我们会对较短序列进行填充，使其能统一放入单个批次张量中

- 从概念上讲，这使得批量生成比单提示生成更难实现
- 在主要章节中，我们使用了 `reasoning_from_scratch.qwen3` 中的 `Qwen3Model` 类（该类使用了附录 C 中解释的 Qwen3 实现）
- 对于批量生成，由于需要跟踪填充标记等，`reasoning_from_scratch.qwen3_batched` 中有一个单独的 `Qwen3Model` 类（源代码可在补充材料中查看：https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py）

- 为演示批量生成工具的用法，我们来看一个具体示例
- 首先从一个单序列文本生成示例开始，这与我们在主章节中使用的示例类似
- 在此，我们将它依次应用于两个提示（`["2+2?", "3+3=6?"]`）：

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 下面我们将使用来自 `reasoning_from_scratch.qwen3_batched` 的类似代码，该代码支持批处理
- 需要注意的是，批处理版本不支持流式输出，这意味着我们必须等待所有结果生成完毕后才能进行解码和打印
- 此处批处理生成使用左填充，具体原理将在下一节中解释
- 现在，让我们先通过一个使用示例来说明其用法（在深入探讨内部工作原理之前）

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 如我们所见，结果与之前完全相同
- 区别在于这些结果是通过 `generate_text_basic_batched_cache` 并行生成的
- 下一节将简要说明其底层原理

- 一种更优化的代码实现将 `generate_text_basic_batched_cache` 替换为 `generate_text_basic_batched_cache_stop`
- `generate_text_basic_batched_cache` 在每个解码步骤中保留活跃批次中的每一行
- `generate_text_basic_batched_cache_stop` 会从活跃计算批次中移除已完成的行（内部实现更复杂，但能优化性能）
- 下图展示了这一过程

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- 旁注：在Qwen3中，`<eos>` 标记实际上是 `

&nbsp;
## E.3 填充与注意力掩码

- 在单样本模式下，如果我们对一个简短的提示（如 `"2+2?"`）进行分词化，可以将其作为形状为 `(1, 4)` 的简单张量传递给模型：
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- 在内部，模型会构建一个标准的因果注意力掩码，使得每个位置只能关注自身及之前的词元
- 如果您不熟悉自注意力机制，我有一篇文章提供了更多背景信息：https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- 从概念上讲，该掩码如下所示：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` 表示“被遮蔽”，`0` 表示“允许”
- 因此第一个标记无法看到后续位置，第二个标记只能查看前两个位置，依此类推
- 这是标准的自回归遮蔽模式

- 批处理改变了这一情况，因为不同的提示通常具有不同的长度
- 假设我们同时处理 `"2+2?"` 和稍长的提示 `"3+3=6?"`
- 由于 PyTorch 张量必须是矩形的，较短的行需要进行填充以匹配较长的行
- 这里采用的是左填充方式：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- 请注意，我们内部会额外保留一个 `attn_mask`；这仅用于追踪填充位置
- 在此 `attn_mask` 中，`True` 表示已填充，`False` 表示未填充
- 我们使用这个额外的 `attn_mask` 来识别因果掩码中对应填充标记 ID 的标记
- 对填充键进行掩码处理以及将填充查询归零是确保批处理行为与单样本执行保持一致的重要步骤

- 顺便一提，我们使用 `<tool_call>` token，但这并不重要，因为对应的 token 位置都会被忽略。

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 第三章：批处理 MATH-500 评估

- 补充材料包含第3章中实现的评估方法脚本，我们可以下载并使用，类似于第6章中的做法：

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- 然后，要运行它，我们可以在代码终端中执行以下命令（如果你不是uv用户，请将 `uv run` 替换为 `python`）：

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- 额外材料还包括一个批处理生成的版本，该版本应用了我们之前讨论过的批处理方法
- 下载方式与之前类似，只是将 `evaluate_math500.py` 替换为 `evaluate_math500_batched.py`

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- 用法也与非批处理版本类似，不同之处在于我们现在提供了一个额外的 `--batch_size` 参数，用于指定大语言模型应并行处理多少个提示和回答。

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- 理想的批处理大小取决于硬件处理能力；批处理大小为64时大约占用23.39 GB内存（非批处理脚本约占用1.84 GB内存）
- 我们将在附录末尾对比并讨论性能差异

&nbsp;
## E.5 第四章：批处理自洽性采样

- 第4章中实现自一致性采样的可选脚本 `self_consistency_math500_batched.py` 不会将不同提示混合到单个填充张量中
- 相反，它会将相同提示重复 `num_samples` 次，并行采样多个续写结果进行自一致性投票
- 由于每行都从相同提示长度开始，该脚本使用 reasoning_from_scratch.qwen3 中的常规 `Qwen3Model`，而非 reasoning_from_scratch.qwen3_batched，因为相同提示长度无需填充

我们可以按如下方式下载脚本：

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- 要下载非批处理版本，只需在上述文件名中删除 `"_batched"` 部分即可
- 我们可以按如下方式运行脚本（非批处理脚本的语法完全相同）

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- 关于性能的更多信息，请参见本附录末尾

&nbsp;
## E.6 第6章：批量 GRPO rollouts

- 第5章中的自我优化是一种顺序技术，其本身无法从批处理中受益
- 可以并行运行多个输入的自我优化循环，但实现起来较为复杂，因此未包含在补充材料中
- 我们将在第6章继续使用批处理版本的RLVR
- 第6章中，我们对不同的rollout使用相同的提示；因此这里不需要填充；与E.5节类似，代码使用来自`reasoning_from_scratch.qwen3`的常规`Qwen3Model`类
- 相关脚本可通过以下方式获取：

In [ ]:
# Non-batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# Batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# Batched version with GPU support
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- 在当前脚本中，`--batch_size` 控制每个步骤内并行生成的轨迹数量
- 这会提高吞吐量，但也会增加内存压力，因此在实践中可能需要减少 `--num_rollouts` 或 `--max_new_tokens` 的值
- 若使用多GPU，FSDP变体遵循相同模式并新增 `--num_gpus` 参数
- 我们将在本附录末尾再次讨论性能相关问题
- 截至本文撰写时，第7章脚本的批处理版本尚未在补充材料中提供，但会逐步添加；从概念上讲，其工作原理将与第6章脚本类似

&nbsp;
## E.7 第8章：批量蒸馏

- 第八章回归第三章的填充感知风格，因为蒸馏示例的提示和答案长度各不相同
- 您可以按如下方式下载脚本和示例训练数据集：

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- 对于非批处理版本，请移除文件名中的 `"_batched"`
- 我们可以按如下方式运行脚本：

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 单序列与批量生成

- 下表总结了上述脚本的运行时间和内存使用情况

| 行号 | 脚本名称                                 | 批次大小 | 内存占用 | H100 总耗时 (分钟) | DGX Spark 总耗时 (分钟) |
|------|------------------------------------------|----------|----------|---------------------|--------------------------|
| 1    | evaluate_math500.py                      | -        | 1.8 GB   | 90.0                | 174.7                    |
| 2    | evaluate_math500_batched.py              | 64       | 23.39 GB | 16.0                | 108.4                    |
|      |                                          |          |          |                     |                          |
| 3    | self_consistency_math500.py              | -        | 1.79 GB  | 252.0               | 340.8                    |
| 4    | self_consistency_math500_batched.py      | 3        | 2.45 GB  | 129.0               | 243.3                    |
|      |                                          |          |          |                     |                          |
| 5    | rlvr_grpo_original_no_kl.py              | -        | 43.35 GB | 68.0                | 63.7                     |
| 6    | rlvr_grpo_original_no_kl_batched.py      | 4        | 44.91 GB | 19.0                | 23.1                     |
|      |                                          |          |          |                     |                          |
| 7    | distill.py                               | -        | 8.29 GB  | 10.9                | 32.8                     |
| 8    | distill_batched.py                       | 4        | 8.34 GB  | 9.1                 | 28.2                     |